In [ ]:
import math
import tkinter as tk
from tkinter import messagebox, ttk

from calculator import CalculatorError, evaluate_expression, format_result


class ScientificCalculatorGUI:
    def __init__(self, root: tk.Tk) -> None:
        self.root = root
        self.root.title("Scientific Calculator")
        self.root.minsize(620, 500)

        self.expression = tk.StringVar()
        self.status = tk.StringVar(value="Enter an expression and press =")
        self.mode = tk.StringVar(value="RAD")
        self.ans = 0.0

        self._build_ui()

    def _build_ui(self) -> None:
        container = ttk.Frame(self.root, padding=12)
        container.grid(row=0, column=0, sticky="nsew")
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)

        container.columnconfigure(0, weight=1)
        container.rowconfigure(3, weight=1)

        title = ttk.Label(container, text="Scientific Calculator", font=("Helvetica", 18, "bold"))
        title.grid(row=0, column=0, sticky="w")

        input_row = ttk.Frame(container)
        input_row.grid(row=1, column=0, sticky="ew", pady=(10, 8))
        input_row.columnconfigure(0, weight=1)

        self.entry = ttk.Entry(input_row, textvariable=self.expression, font=("Menlo", 16), justify="right")
        self.entry.grid(row=0, column=0, sticky="ew")
        self.entry.bind("<Return>", self._on_enter)
        self.entry.bind("<KP_Enter>", self._on_enter)
        self.entry.bind("<Escape>", self._on_escape)

        status_label = ttk.Label(container, textvariable=self.status, font=("Helvetica", 11))
        status_label.grid(row=2, column=0, sticky="w", pady=(0, 8))

        button_frame = ttk.Frame(container)
        button_frame.grid(row=3, column=0, sticky="nsew")

        for column in range(5):
            button_frame.columnconfigure(column, weight=1)
        for row in range(8):
            button_frame.rowconfigure(row, weight=1)

        rows = [
            ["7", "8", "9", "/", "sqrt("],
            ["4", "5", "6", "*", "log("],
            ["1", "2", "3", "-", "ln("],
            ["0", ".", "(", ")", "+"],
            ["sin(", "cos(", "tan(", "^", "%"],
            ["asin(", "acos(", "atan(", "//", "ans"],
            ["pi", "e", "ncr(", "npr(", "factorial("],
            ["Back", "Clear", "AC", "Mode", "="],
        ]

        for row_index, row_values in enumerate(rows):
            for col_index, label in enumerate(row_values):
                button = ttk.Button(
                    button_frame,
                    text=label,
                    command=lambda value=label: self._on_button_press(value),
                )
                button.grid(row=row_index, column=col_index, sticky="nsew", padx=4, pady=4)

        info_frame = ttk.Frame(container)
        info_frame.grid(row=4, column=0, sticky="ew", pady=(10, 0))
        info_frame.columnconfigure(0, weight=1)

        self.mode_label = ttk.Label(info_frame, text="Trig Mode: RAD")
        self.mode_label.grid(row=0, column=0, sticky="w")

        constants_button = ttk.Button(info_frame, text="Constants", command=self._show_constants)
        constants_button.grid(row=0, column=1, sticky="e")

        self.entry.focus_set()

    def _on_enter(self, _: tk.Event) -> None:
        self._evaluate()

    def _on_escape(self, _: tk.Event) -> None:
        self.expression.set("")
        self.status.set("Expression cleared")

    def _on_button_press(self, label: str) -> None:
        if label == "=":
            self._evaluate()
            return
        if label == "Back":
            self._backspace()
            return
        if label == "Clear":
            self.expression.set("")
            self.status.set("Expression cleared")
            return
        if label == "AC":
            self.expression.set("")
            self.ans = 0.0
            self.status.set("Expression and ans reset")
            return
        if label == "Mode":
            self._toggle_mode()
            return

        insertion = "**" if label == "^" else label
        current = self.expression.get()
        self.expression.set(current + insertion)
        self.entry.icursor(tk.END)
        self.entry.focus_set()

    def _evaluate(self) -> None:
        expression = self.expression.get().strip()
        if not expression:
            self.status.set("Enter an expression first")
            return

        try:
            result = evaluate_expression(expression, self.ans, self.mode.get())
        except (CalculatorError, ValueError, ZeroDivisionError, OverflowError) as exc:
            self.status.set(f"Error: {exc}")
            messagebox.showerror("Calculation Error", str(exc))
            return

        self.ans = result
        formatted = format_result(result)
        self.expression.set(formatted)
        self.status.set(f"= {formatted}")

    def _backspace(self) -> None:
        current = self.expression.get()
        if not current:
            return
        self.expression.set(current[:-1])
        self.entry.icursor(tk.END)

    def _toggle_mode(self) -> None:
        self.mode.set("DEG" if self.mode.get() == "RAD" else "RAD")
        self.mode_label.config(text=f"Trig Mode: {self.mode.get()}")
        self.status.set(f"Trigonometric mode set to {self.mode.get()}")

    def _show_constants(self) -> None:
        messagebox.showinfo(
            "Constants",
            (
                f"pi = {math.pi}\n"
                f"e = {math.e}\n"
                f"tau = {math.tau}\n"
                f"ans = {format_result(self.ans)}"
            ),
        )


def main() -> None:
    root = tk.Tk()
    ScientificCalculatorGUI(root)
    root.mainloop()


if __name__ == "__main__":
    main()

In [11]:
import sys
sys.path.insert(0, "/Users/chetanjadhav/Documents/AI voice/Python project/AI voice base")
from calculator import CalculatorError, evaluate_expression, format_result
